In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_06_empirical_potentials.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Atomic Scale Simulations · Lesson 06: Interatomic Potentials — From Born-Oppenheimer to LJ, EAM, Stillinger-Weber & Force Fields

_Session b5e01a74 · Estimated time: ~115 minutes · 46 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Explain the Born-Oppenheimer adiabatic approximation, its $\mathcal{O}(m_e/M_I)^{3/2}$ error scaling, Hartree atomic units, and the trade-off between ab initio and semi-empirical potential energy surfaces.
- Derive key properties of the Lennard-Jones 12-6, Morse, and Born pair potentials, apply LJ reduced units and the Law of Corresponding States, and analyze cubic-spline lookup tables.
- Diagnose the four classic failures of pair potentials in metals ($E_c/T_m$, $E_v/E_c$, the Cauchy relation $C_{12}=C_{44}$, and inward surface relaxation) and show how the Embedded Atom Method (EAM) resolves them via $F(\rho) \propto -\sqrt{\rho}$.
- Analyze cluster potentials and functionals—including Modified EAM (MEAM) and the Stillinger-Weber 3-body potential for tetrahedral Silicon ($\cos\theta_0 = -1/3$).
- Distinguish bonded (2-body stretch, 3-body bend, 4-body dihedral torsion) and non-bonded interactions in organic force fields (AMBER/CHARMM), rigid water models (TIP3P/TIP4P/TIP5P), and polarizable Core-Shell ionic models.


In [2]:
ws.render_section('sec_bo_and_atomic_units')

## 1. The Electronic-Structure Problem, Atomic Units & The Born-Oppenheimer Approximation

Before running any Molecular Dynamics or Monte Carlo simulation, we must specify the **model** governing how atoms interact. Fundamentally, collections of $N_e$ electrons (coordinates $\mathbf{r}_i$, mass $m_e$, charge $-e$) and $N_I$ atomic nuclei (coordinates $\mathbf{R}_I$, mass $M_I$, charge $+Z_I e$) obey the non-relativistic many-body Schrödinger equation.

### Hartree Atomic Units ($\hbar = m_e = e = 4\pi\varepsilon_0 = 1$)
To eliminate fundamental constants from the quantum Hamiltonian, electronic-structure physics uses **Hartree atomic units**:
- **Length (1 Bohr radius $a_0$):**
 $$a_0 = \frac{\hbar^2}{m_e e^2} \approx 0.52918\text{ Å} = 5.29 \times 10^{-9}\text{ cm}$$
- **Energy (1 Hartree $\text{Ha}$):**
 $$1\text{ Ha} = \frac{e^2}{a_0} = 2\text{ Ry} \approx 27.211\text{ eV} \approx 315,775\text{ K} \approx 316,000\text{ K}$$
- **Room-temperature accuracy challenge:** Thermal energy near room temperature ($T \sim 100\text{–}300\text{ K}$) is only $\sim 0.3\text{–}1.0\text{ mHa} \approx 0.01\text{–}0.025\text{ eV}$. Because total electronic energies of heavy atoms are hundreds or thousands of Hartrees, predicting room-temperature phase stability directly from quantum mechanics requires **many decimal places of cancellation**!

In atomic units, the total Hamiltonian splits into an **electronic Hamiltonian** $H_e[\mathbf{R}_I]$ and an **ionic Hamiltonian** $H_I$:
$$H = \underbrace{-\sum_{i=1}^{N_e}\frac{1}{2}\nabla_i^2 + \sum_{i<j}\frac{1}{r_{ij}} - \sum_{i=1}^{N_e}\sum_{I=1}^{N_I}\frac{Z_I}{r_{iI}}}_{H_e[\mathbf{R}_I]\text{ (electron KE + e-e + e-ion)}} \;\;+\;\; \underbrace{-\sum_{I=1}^{N_I}\frac{m_e}{2M_I}\nabla_I^2 + \sum_{I<J}\frac{Z_I Z_J}{R_{IJ}}}_{H_I\text{ (ion KE + ion-ion)}}$$

### The Born-Oppenheimer (1927) Adiabatic Approximation
Because even the lightest nucleus (a proton) is **$M_p / m_e \approx 1836$ times heavier** than an electron, electrons move orders of magnitude faster than nuclei and relax almost instantaneously to their quantum ground state for any instantaneous nuclear configuration $\{\mathbf{R}_I\}$:
1. **Factor the total wavefunction:** $\Psi(\mathbf{r}_i, \mathbf{R}_I) \approx \psi(\mathbf{r}_i \mid \mathbf{R}_I)\,\phi(\mathbf{R}_I)$ (the **adiabatic approximation**).
2. **Solve the electronic ground state at clamped nuclear positions $\mathbf{R}_I$:**
 $$H_e[\mathbf{R}_I]\,\psi(\mathbf{r}_i \mid \mathbf{R}_I) = E_{\text{BO}}(\mathbf{R}_I)\,\psi(\mathbf{r}_i \mid \mathbf{R}_I)$$
3. **Eliminate electrons from the nuclear dynamics:** The nuclei move on a single effective **Potential Energy Surface (PES)**:
 $$V(\mathbf{R}_I) = E_{\text{BO}}(\mathbf{R}_I) + \sum_{I<J}\frac{Z_I Z_J}{R_{IJ}}, \qquad \left(-\sum_I \frac{m_e}{2M_I}\nabla_I^2 + V(\mathbf{R}_I)\right)\phi(\mathbf{R}_I) = E_{\text{total}}\phi(\mathbf{R}_I)$$
- **Accuracy of BO:** The error in total energy from neglecting non-adiabatic electron-phonon coupling scales as:
 $$\text{error}(E_{\text{total}}) \sim \mathcal{O}\!\left(\frac{m_e}{M_I}\right)^{3/2} E_{\text{BO}} \le 10^{-5}\text{ Ha}$$
- **Important subtlety:** The Born-Oppenheimer approximation **does NOT require classical ions**—$\phi(\mathbf{R}_I)$ is still a nuclear wavefunction (allowing quantum zero-point motion and tunneling in Path-Integral MD). Classical MD makes a *second* approximation by replacing the nuclear Schrödinger equation with Newton's law $M_I \ddot{\mathbf{R}}_I = -\nabla_I V(\mathbf{R}_I)$.

### Three Paradigms for Obtaining $V(\mathbf{R}_I)$
1. **Semi-empirical potentials:** Assume a physically motivated functional form (2-body, 3-body, EAM) and fit parameters to experimental data (lattice constants, cohesive energy, elastic moduli, defect/surface energies, gas virial coefficients) and/or DFT calculations. *Fast ($\sim 10^6\text{–}10^9$ atoms), but limited by **transferability** (extrapolating to environments far from the fitting set—GIGO: "garbage in, garbage out").*
2. **Quantum chemistry surface fitting:** Compute $E_{\text{BO}}$ at selected geometries via high-level quantum chemistry and fit an analytic surface. *Hard in high dimensions!*
3. **Ab initio MD (First-Principles):** Solve the electronic structure (e.g., Kohn-Sham DFT) **on the fly** at every MD timestep to obtain forces without any predetermined analytic potential. *Highly transferable, but computationally expensive ($N \sim 10^2\text{–}10^3$ atoms, $t \sim 10\text{–}100\text{ ps}$).*


In [3]:
ws.render_section('q01_bo_mass_ratio_and_error')

**Q1 · easy** · 💡

What physical small parameter justifies the Born-Oppenheimer (1927) adiabatic separation of electronic and nuclear motion, and how does the resulting error in the total energy scale?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check of the reference notes for the mass ratio and exponent.

</div>
</details>

- [ ] **(A)** The electron-to-nuclear mass ratio $m_e / M_I \le 1/1836 \ll 1$, giving an energy error scaling as $\mathcal{O}(m_e / M_I)^{3/2} E_{\text{BO}} \le 10^{-5}\text{ Ha}$
- [ ] **(B)** The fine-structure constant $\alpha \approx 1/137$, giving a relativistic error of $\mathcal{O}(\alpha^2)$
- [ ] **(C)** The ratio of thermal energy to Hartree energy $k_B T / \text{Ha}$, giving an error linear in temperature $T$
- [ ] **(D)** The ratio of Bohr radius to box length $a_0 / L$, which vanishes only under Periodic Boundary Conditions

In [4]:
ws.render_section('q02_bo_classical_ions_misconception')

**Q2 · medium** · 💡

Does the Born-Oppenheimer approximation itself require that the atomic nuclei behave as **classical point particles** obeying $\mathbf{F} = M\mathbf{a}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the bottom box of the reference notes: 'Does not require classical ions (though MD uses that).'

</div>
</details>

- [ ] **(A)** Yes; factoring $\Psi(\mathbf{r}_i, \mathbf{R}_I) = \psi(\mathbf{r}_i | \mathbf{R}_I)\phi(\mathbf{R}_I)$ mathematically forces $\phi(\mathbf{R}_I)$ to be a classical Dirac delta function
- [ ] **(B)** No; Born-Oppenheimer only replaces the electronic degrees of freedom with an effective potential energy surface $E_{\text{BO}}(\mathbf{R}_I)$ in the nuclear Schrödinger equation—treating nuclei classically is a separate approximation made in classical MD
- [ ] **(C)** Yes; quantum nuclei cannot experience an effective potential energy surface
- [ ] **(D)** Only for hydrogen and helium; heavier nuclei become strictly classical under the Born-Oppenheimer approximation

In [5]:
ws.render_section('q03_atomic_units_conversion')

**Q3 · easy** · 💡

In Hartree atomic units ($\hbar = m_e = e = 1$), $1\text{ Hartree (Ha)} = 27.211\text{ eV}$. If an ab initio calculation reports a vacancy migration barrier of $\Delta E = 0.050\text{ Ha}$ in a metal, what is this barrier in $\text{eV}$? (Enter a number to two decimal places, e.g., `1.36`.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply the Hartree-to-eV conversion factor to the given energy barrier.

</div>
</details>

_Numeric answer required._

In [6]:
ws.render_section('q04_room_temp_millihartree')

**Q4 · easy** · 💡

According to the principles of simulation methodology,, why is solving the electronic-structure problem directly to predict room-temperature phase stability ($\sim 100\text{–}300\text{ K}$) so computationally demanding?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare $100\text{ K}$ in eV and Hartrees against the total electronic energy of an atom.

</div>
</details>

- [ ] **(A)** The Bohr radius $a_0 = 0.529\text{ Å}$ is larger than macroscopic simulation boxes
- [ ] **(B)** Room temperature ($300\text{ K}$) exceeds $10\text{ Hartrees}$, causing all core electrons to ionize into a plasma
- [ ] **(C)** $100\text{ K}$ corresponds to only $\approx 0.3\text{ mHa}$ ($0.01\text{ eV}$), whereas total electronic energies are tens to thousands of Hartrees ($1\text{ Ha} \approx 316,000\text{ K}$), requiring extreme relative precision across many decimal places
- [ ] **(D)** The Coulomb interaction between nuclei vanishes at $T = 100\text{ K}$

In [7]:
ws.render_section('q05_three_potential_paradigms')

**Q5 · easy** · 💡

Which simulation approach computes the electronic quantum ground state **\"on the fly\"** at every step along the trajectory as the nuclei move, avoiding the need to postulate or fit an analytic functional form for $V(\mathbf{R})$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare semi-empirical, quantum chemistry surface fitting, and ab initio approaches of the reference notes.

</div>
</details>

- [ ] **(A)** Hard-sphere Monte Carlo
- [ ] **(B)** Semi-empirical pair potentials (e.g., Lennard-Jones 12-6)
- [ ] **(C)** Cubic-spline lookup tables of Morse potentials
- [ ] **(D)** Ab initio Molecular Dynamics (e.g., Born-Oppenheimer / Car-Parrinello MD)

In [8]:
ws.render_section('q06_transferability_definition')

**Q6 · easy** · 💡

In the context of semi-empirical interatomic potentials, what does **transferability** mean?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about interpolation vs. Extrapolation.

</div>
</details>

- [ ] **(A)** The ability of a potential fitted to a limited set of reference properties (e.g., bulk $T = 0$ crystal data) to accurately predict behavior in very different local environments (e.g., surfaces, dislocations, liquids, or high pressures).
- [ ] **(B)** The ability to port the potential's C++ source code to a GPU without syntax errors
- [ ] **(C)** The exact conservation of linear momentum under Periodic Boundary Conditions
- [ ] **(D)** The guarantee that adding more fitting parameters always reduces extrapolation error to zero

In [9]:
ws.render_section('q07_statistical_vs_systematic_error')

**Q7 · medium** · 💡

As summarized of the reference notes ("Which approach to use?"), the total error in an atomistic simulation is a combination of **statistical error** and **systematic error**. How are these two error sources governed?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Which error decreases with more MD timesteps, and which error is inherent to the interatomic potential $V(\mathbf{R})$?

</div>
</details>

- [ ] **(A)** Statistical error is caused by the Born-Oppenheimer approximation, whereas systematic error vanishes as $N_{\text{steps}}^{-1/2}$
- [ ] **(B)** Statistical error is governed by finite sampling (the number of time steps $N_{\text{steps}}$), whereas systematic error is governed by inaccuracies in the interatomic potential $V(\mathbf{R})$
- [ ] **(C)** Running a simulation with an inaccurate empirical potential for $10^{12}$ time steps eliminates systematic error
- [ ] **(D)** Ab initio MD has zero statistical error even for a 10-femtosecond trajectory

In [10]:
ws.render_section('sec_pair_potentials_lj_morse')

## 2. Two-Body Pair Potentials: Lennard-Jones 12-6, Reduced Units, Morse, Born & Spline Tables

In a **pair potential**, the total potential energy is a sum purely over two-body distances $r_{ij} = |\mathbf{r}_i - \mathbf{r}_j|$:
$$V(\{\mathbf{r}_i\}) = \sum_{i<j} v(r_{ij})$$
This approximation works well when closed-shell atoms or rigid molecules remain in a non-degenerate electronic ground state that is only weakly perturbed by neighbors—most notably **neutral rare-gas atoms** ($\text{He, Ne, Ar, Kr, Xe}$).

### Physical Origin of Rare-Gas Interactions
- **Short-range repulsion ($r \to 0$):** Overlap of closed-shell electron clouds triggers Pauli exclusion, raising kinetic energy exponentially: $v_{\text{rep}}(r) \propto \exp(-r/c)$.
- **Long-range attraction ($r \gg r_{\text{core}}$):** Second-order quantum perturbation theory between neutral spherical atoms produces the **London dispersion (fluctuating dipole–induced-dipole) interaction**:
 $$v_{\text{disp}}(r) = -\frac{C_6}{r^6} - \frac{C_8}{r^8} - \cdots$$

### The Lennard-Jones (12-6) Potential (1931)
$$v_{\text{LJ}}(r) = 4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^6\right]$$
- **Why exponents 6 and 12?** The $-(\sigma/r)^6$ term has a rigorous physical basis (London dipole–induced-dipole dispersion). The $+(\sigma/r)^{12}$ Pauli repulsion is chosen for **computational convenience** because $(\sigma/r)^{12} = [(\sigma/r)^6]^2$ can be evaluated by squaring the $r^{-6}$ term without calling `exp(-r/c)`!
- **Key geometric landmarks:**
 1. **Zero-crossing (hard wall diameter):** $v_{\text{LJ}}(\sigma) = 0$.
 2. **Equilibrium pair distance $r_{\min}$:** Setting $\frac{dv}{dr} = 0$ gives:
 $$r_{\min} = 2^{1/6}\sigma \approx 1.12246\,\sigma, \qquad v_{\text{LJ}}(r_{\min}) = -\varepsilon$$
 3. **Force:** $F(r) = -\frac{dv}{dr} = \frac{24\varepsilon}{r}\left[2\left(\frac{\sigma}{r}\right)^{12} - \left(\frac{\sigma}{r}\right)^6\right]$.

### LJ Reduced Units & The Law of Corresponding States
By measuring lengths in $\sigma$, energies in $\varepsilon$, and masses in atomic mass $m$:
$$r^* = \frac{r}{\sigma}, \quad E^* = \frac{E}{\varepsilon}, \quad T^* = \frac{k_B T}{\varepsilon}, \quad \rho^* = \rho\sigma^3, \quad P^* = \frac{P\sigma^3}{\varepsilon}, \quad t^* = t\sqrt{\frac{\varepsilon}{m\sigma^2}}$$
the Lennard-Jones potential becomes parameter-free: $v^*(r^*) = 4\left[(r^*)^{-12} - (r^*)^{-6}\right]$.
- **"There is only 1 LJ system!" (Law of Corresponding States):** A single dimensionless simulation of the LJ fluid determines the phase diagram for *all* noble gases ($\text{Ne, Ar, Kr, Xe}$), predicting a universal critical point ($T_c^* \approx 1.31$, $P_c^* \approx 0.12$, $\rho_c^* \approx 0.31$) and triple point ($T_t^* \approx 0.68\text{–}0.70$).

### The Morse Potential (1929)
Because the LJ $12$-$6$ potential has only **2 parameters** ($\varepsilon, \sigma$), once you fit the lattice constant ($r_0$) and cohesive energy ($\varepsilon$), the curvature at the minimum $v''(r_{\min}) = 72\varepsilon / r_{\min}^2$ (and thus the **bulk modulus $B$**) is completely fixed!
The **Morse potential** replaces power laws with exponentials and introduces a **3rd independent parameter $\alpha$** (width/stiffness parameter):
$$v_{\text{Morse}}(r) = \varepsilon\left[e^{-2\alpha(r - r_0)} - 2e^{-\alpha(r - r_0)}\right] = \varepsilon\left[\left(1 - e^{-\alpha(r-r_0)}\right)^2 - 1\right]$$
- Minimum at $r = r_0$ with well depth $v_{\text{Morse}}(r_0) = -\varepsilon$.
- Curvature at $r_0$: $v_{\text{Morse}}''(r_0) = 2\alpha^2\varepsilon$.
- The 3 parameters $(r_0, \varepsilon, \alpha)$ allow simultaneous fitting to **three experimental properties**:
 1. Equilibrium volume / lattice constant: $\left.\frac{dE}{dV}\right|_{V_0} = 0$
 2. Cohesive energy: $E_{\text{cohesive}} = -E(V_0)$
 3. Bulk modulus: $B = -V\left.\frac{dP}{dV}\right|_{V_0} = V\left.\frac{d^2 E}{dV^2}\right|_{V_0}$

### Born Ionic Potential & Table-Driven Spline Potentials
- **Born Potential for Ionic Crystals (e.g., NaCl):**
 $$v(r) = \frac{Z_i Z_j e^2}{r} + \frac{A}{r^n}$$
 Summing over the alternating $\pm 1$ rock-salt lattice introduces the **Madelung constant** ($\alpha_M \approx 1.7476$). Fitting $A$ and $n$ to the experimental $T=0$ lattice constant $a_0$ and cohesive energy yields $n = 8.87$, and checking the un-fitted bulk modulus gives $B_{\text{calc}} = 4.35 \times 10^{11}\text{ dyn/cm}^2$ vs. $B_{\text{exp}} = 2.52 \times 10^{11}\text{ dyn/cm}^2$—illustrating **"You get what you fit for!"**
- **Table-Driven Cubic Splines for Arbitrary Pair Potentials:**
 For any pair potential more complicated than LJ 12-6 (containing exponentials, square roots, or DFT data), MD codes pre-tabulate $V(r)$ and $dV/dr$ on a radial grid at startup. During the force loop, each distance $r_{ij}$ is mapped to a grid index and interpolated with a **cubic polynomial**, trading expensive transcendental math for a fast **memory fetch + a few FLOPs** (with the minor disadvantage of potential CPU cache misses).


In [11]:
ws.render_section('concept_lj_vs_morse')

### Interactive Explorer: Lennard-Jones 12-6 vs. Morse Potential & Force

Adjust the dimensionless Morse stiffness parameter $\alpha r_0$ and toggle between potential energy $V(r)/\varepsilon$ and interatomic force $F(r)r_0/\varepsilon$. Observe how $\alpha r_0$ tunes the well curvature (bulk modulus $B$) independently of $r_0$ and $\varepsilon$.

**Key Takeaway:** At $\alpha r_0 = 6.0$, the Morse potential has the exact same harmonic curvature $V''(r_0) = 72\varepsilon / r_0^2$ as the LJ 12-6 potential at $r_0$, but a softer exponential core for $r < r_0$. In cubic metals (Girifalco & Weizer), $\alpha r_0$ typically ranges from $3.5$ to $4.5$, which is much softer than LJ!

In [12]:
ws.render_section('q08_origin_of_r6_and_r12')

**Q8 · easy** · 💡

In the Lennard-Jones 12-6 potential $v(r) = 4\varepsilon\left[(\sigma/r)^{12} - (\sigma/r)^6\right]$ for neutral rare-gas atoms, what is the physical origin of the attractive $-(\sigma/r)^6$ term and the reason for choosing the repulsive $+(\sigma/r)^{12}$ exponent?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review, and 8 of the reference notes.

</div>
</details>

- [ ] **(A)** The $-r^{-6}$ term represents permanent monopole-monopole Coulomb attraction, and $+r^{-12}$ represents gravitational repulsion
- [ ] **(B)** Both $r^{-6}$ and $r^{-12}$ are exact analytical solutions of the Schrödinger equation for two Argon atoms at all distances
- [ ] **(C)** The $-r^{-6}$ attraction arises physically from London dispersion (fluctuating dipole–induced-dipole) forces; the $+r^{-12}$ Pauli core repulsion is chosen for computational speed because $(r^{-6})^2$ avoids evaluating an exponential $\exp(-r/c)$
- [ ] **(D)** The $+r^{-12}$ term is physically softer than $\exp(-r/c)$ at very small $r$

In [13]:
ws.render_section('q09_lj_equilibrium_distance')

**Q9 · easy** · 💡

For Argon, the Lennard-Jones diameter parameter (where $v(\sigma) = 0$) is $\sigma = 3.40\text{ Å}$. What is the equilibrium pair separation $r_{\min}$ (in $\text{Å}$) where the potential reaches its minimum $v(r_{\min}) = -\varepsilon$ and the interatomic force vanishes? (Enter a value to two decimal places, e.g., `3.82`.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Find the radial distance where the first derivative of the Lennard-Jones potential $dv/dr$ equals zero.

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('q10_lj_force_at_sigma')

**Q10 · medium** · 💡

Evaluate the dimensionless Lennard-Jones pair force $F^* = F(r)\frac{\sigma}{\varepsilon} = -\frac{\sigma}{\varepsilon}\frac{dv}{dr}$ at the zero-crossing distance $r = \sigma$.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Differentiate the Lennard-Jones potential with respect to distance to obtain the force expression, then evaluate it at $r = \sigma$.

</div>
</details>

_Numeric answer required._

In [15]:
ws.render_section('q11_corresponding_states_meaning')

**Q11 · easy** · 💡

In the formulation of Lennard-Jones reduced units, the reference notes emphasize that **"There is only 1 LJ system!"** and compares reduced critical and triple-point parameters ($T_c^*, P_c^*, \rho_c^*, T_t^*, P_t^*$) across Neon, Argon, Krypton, and Xenon. What does this principle imply?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Think about what happens when lengths, energies, and temperatures are scaled by $\sigma$ and $\varepsilon$.

</div>
</details>

- [ ] **(A)** Quantum zero-point effects are identical in Helium-4 and Xenon
- [ ] **(B)** Neon, Argon, Krypton, and Xenon all have the exact same melting temperature in Kelvin ($T_m = 84\text{ K}$)
- [ ] **(C)** The Lennard-Jones potential has only one stable phase (liquid) at all temperatures
- [ ] **(D)** When expressed in reduced units ($T^* = k_B T/\varepsilon$, $\rho^* = \rho\sigma^3$, $P^* = P\sigma^3/\varepsilon$), all classical Lennard-Jones substances obey the exact same universal equation of state and phase diagram

In [16]:
ws.render_section('q12_reduced_temperature_calc')

**Q12 · medium** · 💡

According to the Lennard-Jones phase diagram table, the triple-point temperature of the LJ fluid is $T_t^* = k_B T_t / \varepsilon \approx 0.70$. Given that Argon has well depth $\varepsilon / k_B = 119.8\text{ K}$, what is the predicted triple-point temperature $T_t$ of Argon in $\text{K}$? (Enter a value to one decimal place, e.g., `83.9`.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Convert the dimensionless reduced temperature into Kelvin using the characteristic energy well depth.

</div>
</details>

_Numeric answer required._

In [17]:
ws.render_section('q13_aziz_helium_and_h2')

**Q13 · medium** · 💡

Based of the reference notes ("Atomic systems"), select **all** true statements regarding rare-gas and $\text{H}_2$ pair interactions:

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the notes on the Aziz He-He potential and why $\text{H}_2$ behaves almost like a rare gas.

</div>
</details>

- [ ] **(A)** The He–He pair interaction (constructed by Aziz using virial coefficients, transport data, and quantum chemistry) is known to better than $0.1\text{ K}$, but simulating condensed Helium requires quantum simulations due to its tiny mass
- [ ] **(B)** Molecular $\text{H}_2$ behaves almost like a spherical rare-gas atom because its rotational (angular) degree of freedom averages out due to quantum effects, though it has a much larger polarizability
- [ ] **(C)** Three-body and many-body interactions in rare gases are small compared to the two-body term, though not strictly zero
- [ ] **(D)** Helium-4 solidifies at room temperature and atmospheric pressure under classical Lennard-Jones dynamics

In [18]:
ws.render_section('q14_morse_three_parameters')

**Q14 · easy** · 💡

In the Morse potential $v(r) = \varepsilon\left[e^{-2\alpha(r-r_0)} - 2e^{-\alpha(r-r_0)}\right]$, what is the specific role of the third parameter $\alpha$ (compared to the 2-parameter Lennard-Jones potential)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check of the reference notes and the interactive LJ vs. Morse explorer.

</div>
</details>

- [ ] **(A)** It independently controls the width/curvature of the potential well ($v''(r_0) = 2\alpha^2\varepsilon$), allowing the potential to simultaneously fit three properties: lattice constant ($r_0$), cohesive energy ($\varepsilon$), and bulk modulus ($B = V d^2E/dV^2$)
- [ ] **(B)** It makes the potential energy oscillate between positive and negative values at large $r$
- [ ] **(C)** It introduces a 3-body angular dependence on $\cos\theta_{ijk}$
- [ ] **(D)** It forces the cohesive energy $E_{\text{coh}}$ to equal the vacancy formation energy $E_v$

In [19]:
ws.render_section('q15_morse_curvature_matching')

**Q15 · hard** · 💡

At the equilibrium distance $r_0$, the Lennard-Jones 12-6 potential has second derivative $v_{\text{LJ}}''(r_0) = 72\,\varepsilon / r_0^2$, while the Morse potential has second derivative $v_{\text{Morse}}''(r_0) = 2\alpha^2 \varepsilon$. What dimensionless value of $\alpha r_0$ makes the Morse potential have the **exact same harmonic curvature (stiffness)** at $r = r_0$ as the Lennard-Jones 12-6 potential?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Equate the two second-derivative expressions at the equilibrium separation and solve for the dimensionless parameter combination.

</div>
</details>

_Numeric answer required._

In [20]:
ws.render_section('q16_hard_sphere_integration_error')

**Q16 · medium** · 💡

In the discussion of Various Other Empirical Potentials, the **Hard Sphere** potential ($v(r) = \infty$ for $r < \sigma$, $0$ for $r \ge \sigma$, used by Alder & Wainwright in 1957) is described as having **"no integration error"**. Why?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How do hard spheres move between instantaneous elastic collisions?

</div>
</details>

- [ ] **(A)** The hard-sphere potential is infinitely differentiable ($C^\infty$) at $r = \sigma$
- [ ] **(B)** Between collisions, hard spheres experience zero force and travel in exact straight lines ($\mathbf{r}(t) = \mathbf{r}_0 + \mathbf{v}t$), allowing event-driven MD to solve collision times analytically without finite-difference timestep truncation error
- [ ] **(C)** Hard spheres do not exchange momentum during collisions, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(D)** The Verlet algorithm has $\mathcal{O}(\Delta t^{12})$ accuracy for discontinuous step potentials

In [21]:
ws.render_section('q17_born_ionic_potential_nacl')

**Q17 · medium** · 💡

In the discussion of Fit for a Born potential, the parameters $A$ and $n$ in $v(r) = \frac{Z_i Z_j e^2}{r} + \frac{A}{r^n}$ for NaCl are determined by fitting the $T=0$ cohesive energy and lattice constant $a_0$ (yielding $n = 8.87$). When this fitted potential is used to predict the bulk modulus $B$, it gives $4.35 \times 10^{11}\text{ dyn/cm}^2$ compared to the experimental value of $2.52 \times 10^{11}\text{ dyn/cm}^2$. What lesson does this illustrate?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the highlighted takeaway at the bottom of.

</div>
</details>

- [ ] **(A)** The Madelung constant for a simple cubic alternating lattice is zero
- [ ] **(B)** NaCl is a covalent metal that has zero Coulomb interaction, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(C)** "You get what you fit for!" — fitting a two-parameter form to energy and lattice constant ($E$ and $dE/da_0$) does not guarantee accuracy for un-fitted second derivatives like the bulk modulus ($d^2E/da_0^2$)
- [ ] **(D)** A power-law core $A/r^n$ always underestimates the bulk modulus by a factor of 10

In [22]:
ws.render_section('q18_table_lookup_splines')

**Q18 · medium** · 💡

In production MD codes, why are arbitrary pair potentials (and EAM functions) evaluated using **pre-computed lookup tables with cubic polynomial (spline) interpolation** rather than evaluating analytic expressions at every pair distance, and what is the main hardware trade-off?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check of the reference notes ('Arbitrary pair potential').

</div>
</details>

- [ ] **(A)** Advantage: zero memory usage; Disadvantage: forces become discontinuous at every grid point
- [ ] **(B)** Advantage: reduces all-pairs complexity from $\mathcal{O}(N^2)$ to $\mathcal{O}(\log N)$; Disadvantage: only works for Lennard-Jones 12-6
- [ ] **(C)** Advantage: eliminates the need for a radial cutoff $r_c$; Disadvantage: requires solving a $N \times N$ matrix inversion at every step
- [ ] **(D)** Advantage: completely general code that evaluates any potential in a few FLOPs per distance without transcendental functions; Disadvantage: memory fetch latency and potential CPU cache misses

In [23]:
ws.render_section('sec_metals_and_eam')

## 3. Failure of Pair Potentials in Metals & The Embedded Atom Method (EAM / MEAM)

In **metals**, valence electrons are delocalized across the crystal, forming a "sea" of conduction electrons around positively charged ion cores. Because of the **Pauli exclusion principle** and metallic screening, the strength of an individual metallic bond **depends strongly on the local coordination number $Z$** (the local electron density):
- As coordination number $Z$ **increases**, electrons are crowded into higher kinetic-energy states, so **each individual bond becomes weaker**.
- Conversely, when neighbors are removed (at a **free surface** or around a **vacancy**), the remaining bonds become **stronger and shorter**!
- Tight-binding second-moment theory shows that metallic cohesive energy per atom scales approximately as the **square root of coordination number**:
 $$E_{\text{metal}}(Z) \propto -\sqrt{Z} \qquad \text{whereas a pair potential forces} \qquad E_{\text{pair}}(Z) \propto -Z$$

### The 4 Classic Failures of Pair Potentials in Metals
Comparing experimental FCC metals ($\text{Cu, Ag, Pt, Au}$) against rare-gas Argon ($\text{Ar}$) and the Lennard-Jones ($\text{LJ}$) pair potential ( of the reference notes, after Ercolessi 1997):

| Dimensionless Property | Cu | Ag | Pt | Au | Ar (Rare Gas) | LJ Pair Potential |
|:--- |:---: |:---: |:---: |:---: |:---: |:---: |
| **$E_c / (k_B T_m)$** (Cohesive / Melting) | $30$ | $28$ | $33$ | $33$ | $11$ | $13$ |
| **$E_v / E_c$** (Vacancy Formation / Cohesive) | $0.33$ | $0.36$ | $0.26$ | $0.25$ | $0.95$ | $1.0$ |
| **$C_{12} / C_{44}$** (Cubic Shear Elastic Ratio) | $1.5$ | $1.9$ | $3.3$ | $3.7$ | $1.1$ | $1.0$ (Cauchy) |
| **Surface Relaxation Direction** | **Inward** | **Inward** | **Inward** | **Inward** | Outward | **Outward** |

1. **Cohesive-to-Melting Ratio ($E_c / k_B T_m \approx 30$ vs. $13$):** In a metal, a huge cohesive energy comes from the uniform volume-dependent electron gas (which doesn't resist shear/melting), whereas pair potentials tie all cohesion to pair bonds.
2. **Vacancy Formation Energy ($E_v / E_c \approx \frac{1}{4}\text{–}\frac{1}{3}$ vs. $1$):** In an unrelaxed pair potential, removing one atom from bulk ($Z=12$) and placing it on the surface breaks $Z$ bonds and reforms $Z/2$ bonds, costing $E_v = E_c$. In a metal ($E \propto -\sqrt{Z}$), the 12 neighbors around the vacancy drop from $Z=12$ to $Z=11$, **strengthening their remaining bonds** and lowering $E_v$ to:
 $$\frac{E_v}{E_c} \approx 12\left(1 - \sqrt{\frac{11}{12}}\right) \approx 0.51 \quad (\text{and } \approx 0.25\text{–}0.35 \text{ after lattice relaxation!})$$
3. **Elastic Constants & The Cauchy Relation ($C_{12} = C_{44}$):** For *any* central two-body pair potential in a cubic crystal where every atom sits at a center of inversion symmetry (at $P=0$), elasticity theory proves the **Cauchy relation $C_{12} = C_{44}$** ($C_{12}/C_{44} = 1$). Real cubic metals violate this strongly ($C_{12}/C_{44} \approx 1.5\text{–}3.7$) because the electron gas resists volume change ($C_{12}$) much more than volume-conserving shear ($C_{44}$).
4. **Inward Surface Relaxation:** At a free metal surface, under-coordinated surface atoms ($Z_{\text{surf}} < 12$) form stronger, shorter bonds with the subsurface layer, causing the top interlayer spacing to **contract inward**. Pair potentials Have constant bond strength and lose attractive 2nd-neighbor forces across the surface, falsely predicting **outward expansion**.

### Taxonomy: Overcoming Deficiencies of Pair Potentials
- **Pair potentials $\to$ Pair functionals** (via *Non-linearity*): e.g., **EAM** / Finnis-Sinclair / Glue models.
- **Pair potentials $\to$ Cluster potentials** (via *Many-body explicit angles*): e.g., **Stillinger-Weber**.
- **Cluster potentials + Non-linearity $\to$ Cluster functionals**: e.g., **MEAM**, Tersoff, Brenner.

### The Embedded Atom Method (EAM — Daw & Baskes, 1984)
EAM models the total energy as the sum of a non-linear **embedding energy** $F_i(\rho_i)$ required to place atom $i$ into the local host electron density $\rho_i$, plus a short-range repulsive core-core **pair potential** $\phi_{ij}(R_{ij})$:
$$E_{\text{tot}} = \sum_i F_i(\rho_i) + \frac{1}{2}\sum_{i \ne j}\phi_{ij}(R_{ij}), \qquad \rho_i = \sum_{j \ne i} f_j(R_{ij})$$
- **Three 1D functions to optimize** for a single element: pair repulsion $\phi(r)$, atomic electron density tail $f(r)$, and embedding function $F(\rho)$.
- Because $F(\rho)$ is concave upward ($F'(\rho) < 0, F''(\rho) > 0$, roughly $F(\rho) \propto -\sqrt{\rho}$), the incremental energy gained per additional neighbor diminishes with increasing $\rho$.
- **Computational cost:** Even though EAM is a many-body potential, it requires only **two radial passes** over the standard neighbor list (Pass 1: accumulate $\rho_i = \sum_j f(R_{ij})$ and compute $F'(\rho_i)$; Pass 2: compute forces), so it remains **$\mathcal{O}(N)$ and only $\sim 2\times$ the cost of a simple pair potential**!
- **Limitations of standard EAM & Modified EAM (MEAM):** Standard EAM uses spherically symmetric density tails $f(R_{ij})$, making it ideal for late FCC transition metals ($\text{Cu, Ag, Au, Ni, Pd, Pt}$), but inadequate for covalent semiconductors ($\text{Si}$), early BCC/HCP transition metals ($\text{Ti, Mo, W, Cr}$), or metals like $\text{Al}$ with strong directional charge distortion under shear. **Modified EAM (MEAM, Baskes)** augments the background density with 3-body angular terms:
 $$\rho_i = \sum_{j,k} f(R_{ij})\,f(R_{ik})\,g(\cos\theta_{ijk})$$


In [24]:
ws.render_section('concept_eam_metals')

### Interactive Explorer: Pair Potentials ($-Z$) vs. EAM ($-Z^p$) in Metals

Adjust the EAM embedding non-linearity exponent $p$ ($E(Z) \propto -Z^p$, where $p = 0.5$ corresponds to $F(\rho) \propto -\sqrt{\rho}$ and $p = 1.0$ recovers a linear pair potential) and toggle between Cohesive Energy per Atom and Relative Bond Strength.

**Key Takeaway:** When $p = 0.5$ ($E \propto -\sqrt{Z}$), breaking a bond strengthens the remaining bonds ($\propto Z^{-1/2}$). This single physical feature explains both why $E_v / E_c \ll 1$ and why under-coordinated metal surfaces ($Z=8$) contract inward!

In [25]:
ws.render_section('q19_four_failures_table_matching')

**Q19 · medium** · 💡

Based on the Ercolessi comparison table, select **all** statements that accurately contrast real FCC metals ($\text{Cu, Ag, Pt, Au}$) with Lennard-Jones ($\text{LJ}$) pair potentials:

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check all four rows: $E_c/T_m$, $E_v/E_c$, $C_{12}/C_{44}$, and surface relaxation.

</div>
</details>

- [ ] **(A)** In real FCC metals, $E_c / T_m \approx 28\text{–}33$, whereas a Lennard-Jones pair potential gives $E_c / T_m \approx 13$ (similar to Argon at $11$)
- [ ] **(B)** In real FCC metals, the vacancy formation energy is only $E_v / E_c \approx 0.25\text{–}0.36$, whereas an unrelaxed pair potential predicts $E_v / E_c = 1.0$
- [ ] **(C)** In real FCC metals, $C_{12} / C_{44} \approx 1.5\text{–}3.7$, whereas any central pair potential in a cubic lattice forces the Cauchy relation $C_{12} / C_{44} = 1.0$
- [ ] **(D)** At a free surface, real metals expand outward whereas a Lennard-Jones pair potential predicts inward contraction

In [26]:
ws.render_section('q20_pair_potential_vacancy_derivation')

**Q20 · medium** · 💡

In a crystal with nearest-neighbor coordination number $Z = 12$ modeled by a nearest-neighbor pair potential of bond energy $-\epsilon_b$, calculate the unrelaxed vacancy formation energy ratio $E_v / E_c$, where $E_c$ is the bulk cohesive energy per atom and $E_v$ is the net energy cost of removing an atom from the bulk to a surface kink site.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Account for the bulk bonds broken to extract the atom versus the bonds reformed when re-adsorbing it at a kink site, and compare this net energy to the cohesive energy per atom.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('q21_sqrt_z_vacancy_derivation')

**Q21 · hard** · 💡

Now consider the metallic square-root coordination model where an atom with local coordination $Z$ has binding energy $E(Z) = -E_c \sqrt{Z / Z_0}$, with bulk FCC coordination $Z_0 = 12$. When one bulk atom is removed to a kink site with full bulk coordination, each of its 12 former nearest neighbors drops from $Z = 12$ to $Z = 11$. Calculate the resulting unrelaxed vacancy formation energy ratio $E_v / E_c$ to two decimal places (e.g., `0.51`).

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Sum the change in binding energy across all undercoordinated neighbor atoms whose coordination was reduced, and express the total vacancy energy in units of $E_c$.

</div>
</details>

_Numeric answer required._

In [28]:
ws.render_section('q22_cauchy_relation_metals')

**Q22 · medium** · 💡

In a cubic crystal at zero external pressure where every atom occupies a site with inversion symmetry, any central two-body pair potential $V = \sum_{i<j}\phi(r_{ij})$ mathematically enforces which relationship between the elastic stiffness constants $C_{12}$ and $C_{44}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check of the reference notes.

</div>
</details>

- [ ] **(A)** The Cauchy relation $C_{12} = C_{44}$ ($C_{12} / C_{44} = 1$)
- [ ] **(B)** $C_{12} = 0$
- [ ] **(C)** $C_{12} = 3 C_{44}$
- [ ] **(D)** $C_{12} = -C_{44}$

In [29]:
ws.render_section('q23_metal_surface_inward_relaxation')

**Q23 · medium** · 💡

Why does the outermost atomic layer of a clean metal surface relax **inward** (contract toward the second layer), whereas a Lennard-Jones pair potential predicts **outward** relaxation?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What happens to the electron density $\rho_i$ and the strength of the remaining back-bonds when a surface atom loses its top neighbors?

</div>
</details>

- [ ] **(A)** In a metal, atmospheric pressure pushes the surface atoms inward
- [ ] **(B)** In a metal, losing neighbors at the surface lowers the local electron density $\rho_i$; because $F(\rho)$ is concave ($E \propto -\sqrt{Z}$), the remaining bonds to the subsurface layer become stronger and shorter, pulling the surface layer inward
- [ ] **(C)** In a pair potential, losing second-neighbor attractions across the surface pulls the top layer inward
- [ ] **(D)** Surface atoms in metals have higher coordination number ($Z = 18$) than bulk atoms ($Z = 12$)

In [30]:
ws.render_section('q24_eam_functional_form')

**Q24 · easy** · 💡

In the Embedded Atom Method (Daw & Baskes, 1984), the total potential energy of a monatomic metal is written as:
$$E_{\text{tot}} = \sum_i F(\rho_i) + \frac{1}{2}\sum_{i \ne j}\phi(R_{ij}), \qquad \rho_i = \sum_{j \ne i} f(R_{ij})$$
What do the three functions $f(R_{ij})$, $F(\rho_i)$, and $\phi(R_{ij})$ physically represent?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review and 21 of the reference notes.

</div>
</details>

- [ ] **(A)** $f(R_{ij})$ is a 3-body bond angle cosine; $F(\rho_i)$ is a linear function $F(\rho) = c\rho$; and $\phi(R_{ij}) = 0$
- [ ] **(B)** $f(R_{ij})$ is the pair force; $F(\rho_i)$ is the nuclear kinetic energy; and $\phi(R_{ij})$ is the long-range dipole-dipole dispersion
- [ ] **(C)** $f(R_{ij})$ is the spherically averaged electron density contributed by neighbor $j$ at site $i$; $F(\rho_i)$ is the non-linear embedding energy to place atom $i$ into host electron density $\rho_i$; and $\phi(R_{ij})$ is a short-range electrostatic core-core pair repulsion
- [ ] **(D)** $f(R_{ij})$ is the Fermi-Dirac distribution; $F(\rho_i)$ is the Helmholtz free energy; and $\phi(R_{ij})$ is the Madelung constant

In [31]:
ws.render_section('q25_linear_embedding_reduces_to_pair')

**Q25 · hard** · 💡

Suppose someone chooses a **linear** embedding function $F(\rho_i) = c \rho_i$ (where $c$ is a constant) in the EAM energy expression $E_{\text{tot}} = \sum_i F\!\left(\sum_{j\ne i} f(R_{ij})\right) + \frac{1}{2}\sum_{i\ne j}\phi(R_{ij})$. What happens to the EAM model in this case?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the governing physical expression and substitute the specified parameters.

</div>
</details>

- [ ] **(A)** It violates translational invariance, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(B)** It becomes a 4-body dihedral potential, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(C)** It predicts $E(Z) \propto -\sqrt{Z}$, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(D)** It collapses mathematically into an ordinary two-body pair potential $\frac{1}{2}\sum_{i\ne j} \phi_{\text{eff}}(R_{ij})$ with $\phi_{\text{eff}}(r) = \phi(r) + 2c f(r)$, losing all many-body metallic properties (so $C_{12} = C_{44}$ and $E_v = E_c$ again!)

In [32]:
ws.render_section('q26_potential_taxonomy_classification')

**Q26 · medium** · 💡

According to the 4-box taxonomy of the reference notes (*Pair potentials*, *Pair functionals*, *Cluster potentials*, *Cluster functionals*), how are **EAM**, **Stillinger-Weber (SW)**, and **Modified EAM (MEAM)** classified?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Adding non-linearity $F(\cdot)$ makes a 'functional'; adding explicit 3-body/many-body angles makes a 'cluster' model.

</div>
</details>

- [ ] **(A)** EAM is a Pair Functional; Stillinger-Weber is a Cluster Potential; MEAM is a Cluster Functional
- [ ] **(B)** EAM is a Cluster Potential; Stillinger-Weber is a Pair Functional; MEAM is a Pair Potential
- [ ] **(C)** All three are Pair Potentials
- [ ] **(D)** EAM and Stillinger-Weber are Pair Potentials; only MEAM is a Pair Functional

In [33]:
ws.render_section('q27_eam_best_elements_and_meam')

**Q27 · medium** · 💡

Why does standard EAM work exceptionally well for late FCC transition metals like $\text{Cu, Ag, Au, Ni, Pd, Pt}$, whereas **Modified EAM (MEAM)** with angular density terms $g(\cos\theta_{ijk})$ is needed for $\text{Si}$, $\text{Ti}$, and BCC refractory metals ($\text{Mo, W}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider the symmetry of the local charge density in filled/nearly-filled $d$-shells vs. Partially filled $d$ or $sp^3$ covalent orbitals.

</div>
</details>

- [ ] **(A)** Standard EAM can only be formulated for lattices with 4 atoms per unit cell
- [ ] **(B)** Standard EAM assumes spherically symmetric atomic electron densities $f(R_{ij})$, which is accurate for nearly-filled $d$-band FCC metals, whereas early transition metals, $\text{Al}$ under shear, and covalent $\text{Si}$ have strong directional ($d$ or $sp^3$) bonding requiring angular forces
- [ ] **(C)** Cu and Au have no valence electrons, whereas Si is a free-electron metal
- [ ] **(D)** MEAM removes the embedding function $F(\rho)$ and uses only Coulomb terms

In [34]:
ws.render_section('q28_phonon_dispersion_validation')

**Q28 · easy** · 💡

In empirical potential modeling, phonon dispersion curves $\omega(\mathbf{k})$ for FCC Nickel computed with two EAM potentials (NRL vs. Voter & Chen) are compared against PW91 DFT and room-temperature neutron diffraction data. What general principle of potential validation does this comparison illustrate?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Two EAM potentials with the same functional form can have very different accuracy away from $k \to 0$!

</div>
</details>

- [ ] **(A)** Neutron diffraction can only measure the cohesive energy at $T = 0\text{ K}$
- [ ] **(B)** All EAM potentials predict zero phonon frequencies at the zone boundary X and L points
- [ ] **(C)** Even with the same EAM functional form, different parameterizations vary widely in accuracy across the Brillouin zone (NRL matches neutron diffraction and DFT closely, whereas Voter & Chen overestimates high-frequency phonons), showing why potentials must be tested against independent experiments
- [ ] **(D)** DFT always underestimates phonon frequencies by a factor of 5

In [35]:
ws.render_section('sec_covalent_organic_water_ionic')

### Silicon & The Stillinger-Weber (1985) 3-Body Potential
Crystalline **Silicon** adopts the open **diamond cubic** structure with a low coordination number of **$Z = 4$**, stabilized by directional **$sp^3$ covalent bonds** pointing toward the corners of a regular tetrahedron ($\theta_{\text{tet}} = 109.47^\circ$, where $\cos\theta_{\text{tet}} = -\frac{1}{3}$).
- **Why any pair potential fails for Silicon:** A radially symmetric pair potential always favors maximizing the number of nearest neighbors inside the attractive well, causing diamond cubic ($Z=4$) to collapse spontaneously into a high-density close-packed metallic structure (FCC/HCP with $Z=12$ and $60^\circ / 90^\circ$ bond angles)!
- **Stillinger-Weber (SW) Potential (`Phys. Rev. B 31, 5262 (1985)`):** Combines a 2-body radial term $v_2(r_{ij})$ with a **3-body angular penalty** $v_3(\mathbf{r}_i, \mathbf{r}_j, \mathbf{r}_k)$ summed over all bond triplets $(j, i, k)$ centered at atom $i$ (for $r_{ij}, r_{ik} < a$):
 $$v_{\text{two body}} = \sum_{i<j}\left(B r_{ij}^{-4} - A\right)\exp\!\left[\frac{1}{r_{ij} - a}\right]$$
 $$v_{\text{three body}} = \sum_{i,j,k} \lambda \exp\!\left[\frac{\gamma}{r_{ij} - a} + \frac{\gamma}{r_{ik} - a}\right]\left(\cos\theta_{jik} + \frac{1}{3}\right)^2$$
 - **Tetrahedral selectivity:** Because $\left(\cos\theta_{jik} + \frac{1}{3}\right)^2 \ge 0$, the 3-body term is **zero if and only if $\cos\theta_{jik} = -1/3$ ($\theta_{jik} = 109.47^\circ$)** and strictly positive ($\lambda > 0$) for any non-tetrahedral angle (such as $60^\circ$ or $90^\circ$ in close-packed crystals).
 - **Smooth cutoff at $r = a$:** As $r \to a^-$, $\frac{1}{r - a} \to -\infty$, so $\exp\!\left(\frac{1}{r-a}\right) \to 0$ with **all derivatives vanishing smoothly** at $r = a$!
 - **Empirical fit targets:** Stillinger and Weber parameterized SW to reproduce the **lattice constant, cohesive energy, melting point, and liquid structure** of Silicon (remarkably capturing the fact that Silicon melts into a higher-coordination $Z \approx 6$ metallic liquid that is denser than the solid!).

### Hydrocarbon & Biomolecular Force Fields (AMBER, CHARMM, MM3)
For organic molecules, polymers, and proteins, covalent bonds do not break or form during standard thermal simulations. Force fields therefore separate **intra-molecular (bonded)** and **inter-molecular (non-bonded)** interactions:
$$V_{\text{AMBER}} = \underbrace{\sum_{\text{bonds}} k_r (r_{ij} - r_0)^2}_{\text{2-body bond stretch}} + \underbrace{\sum_{\text{angles}} k_a (\theta_{ijk} - \theta_0)^2}_{\text{3-body angle bend}} + \underbrace{\sum_{\text{dihedrals}} v_n [1 - \cos(n\phi)]}_{\text{4-body dihedral torsion}} + \underbrace{\sum_{i<j}\left[4\varepsilon_{ij}\!\left(\left(\frac{\sigma_{ij}}{r_{ij}}\right)^{12} - \left(\frac{\sigma_{ij}}{r_{ij}}\right)^6\right) + \frac{q_i q_j}{r_{ij}}\right]}_{\text{Non-bonded LJ + Coulomb}}$$
- **4-Body Dihedral Torsion (e.g., Ethane $\text{H}_3\text{C–CH}_3$):** Rotating around the central $\text{C–C}$ bond changes the relative orientation of the two $\text{CH}_3$ groups between lower-energy **staggered** conformations and higher-energy **eclipsed** conformations ($V_{\text{torsion}} = K\cos(3\phi)$), which requires a **4-body potential** defining the angle between two planes $(1\text{–}2\text{–}3)$ and $(2\text{–}3\text{–}4)$.
- **Fixed hybridization limitation (Ethane vs. Ethene):** Because standard harmonic bonds cannot break or change bond order, a single $\text{C–C}$ bond in ethane ($sp^3$) and a double $\text{C=C}$ bond in ethene ($sp^2$) require **different predefined atom types and force constants** (or a reactive bond-order cluster functional like Tersoff/Brenner/ReaxFF).

### Water Potentials (BNS, MCY, ST2, SPC, TIP3P, TIP4P, TIP5P)
Because water's hydrogen-bonding network and anomalous density maximum govern biology and chemistry, specialized rigid point-charge models are widely used:
- **TIP3P / SPC (3 sites):** Charges on $\text{O}$ and two $\text{H}$ atoms, plus a single LJ 12-6 center on Oxygen.
- **TIP4P (4 sites):** Moves the negative Oxygen charge onto a massless 4th dummy site $\text{M}$ along the $\text{H–O–H}$ angle bisector to improve the quadrupole moment.
- **TIP5P (Mahoney & Jorgensen, 2000 — ):** A **5-site rigid molecule** with:
 - **Oxygen at the center** interacting with other Oxygens via **LJ 12-6** (no charge on the O center itself!).
 - **4 point charges ($q = \pm 0.24\,e$)**: $+0.24\,e$ on the two $\text{H}$ atoms ($r_{\text{OH}} = 0.9572\text{ Å}$, $\theta_{\text{HOH}} = 104.52^\circ$) and $-0.24\,e$ on two tetrahedral lone-pair dummy sites ($r_{\text{OL}} = 0.70\text{ Å}$, $\theta_{\text{LOL}} = 109.47^\circ$).

### Charged & Polarizable Ionic Systems (Buckingham + Core-Shell Model)
- **Buckingham + Coulomb Potential:**
 $$V(r) = \underbrace{A\exp\!\left(-\frac{r}{B}\right) - \frac{C}{r^6}}_{\text{Buckingham}} + \underbrace{\frac{q_1 q_2}{r}}_{\text{Coulomb}}$$
- **The Core-Shell Model for Electronic Polarizability (Dick & Overhauser):**
 In ionic oxides and halides (e.g., $\text{MgO}, \text{SiO}_2, \text{BaTiO}_3$), strong local electric fields $\mathbf{E}$ from neighboring ions induce dipole moments $\mathbf{p} = \alpha\mathbf{E}$ on polarizable anions ($\text{O}^{2-}, \text{Cl}^-$). The **Shell Model** splits each ion into two coupled particles:
 1. A massive positively charged **ion core** ($q_{\text{core}}$), and
 2. A light/massless negatively charged **valence electron shell** ($q_{\text{shell}}$),
 connected by a **harmonic spring** $V_{\text{spring}}(d) = \frac{1}{2}k_s d^2$, where the spring constant $k_s$ is inversely proportional to the ion's electronic polarizability ($\alpha = q_{\text{shell}}^2 / k_s$).


In [36]:
ws.render_section('concept_stillinger_weber')

### Interactive Explorer: Stillinger-Weber 3-Body Angular Penalty for Silicon

Vary the 3-body strength parameter $\lambda$ and normalized bond length $r / \sigma$ to see how the Stillinger-Weber angular term $h(r, r, \theta) = \lambda \exp[2\gamma/(r-a)](\cos\theta + 1/3)^2$ penalizes non-tetrahedral bond angles.

**Key Takeaway:** The $(\cos\theta_{jik} + 1/3)^2$ factor creates a steep energy valley centered at $\theta_{\text{tet}} = 109.47^\circ$, imposing heavy energy penalties on the $60^\circ$ and $90^\circ$ angles characteristic of close-packed FCC/BCC structures.

In [37]:
ws.render_section('concept_organic_dihedral')

### Interactive Explorer: 4-Body Dihedral Torsion Potential in Organic Force Fields

Adjust the torsional periodicity $n$ ($n=3$ for $sp^3\text{–}sp^3$ ethane $\text{H}_3\text{C–CH}_3$, $n=2$ for planar conjugated/alkene bonds) and rotational barrier height $V_n$.

**Key Takeaway:** Capturing the energy difference between staggered and eclipsed conformations around a $\text{C–C}$ single bond requires a **4-body dihedral potential** $V(\phi)$ because three atoms only define a single plane.

In [38]:
ws.render_section('q29_why_silicon_needs_3body')

**Q29 · easy** · 💡

Why can't crystalline Silicon (diamond cubic structure, coordination number $Z = 4$) be stabilized by any radially symmetric two-body pair potential?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

What crystal structure maximizes the number of neighbors at the pair potential minimum $r_0$?

</div>
</details>

- [ ] **(A)** Silicon atoms have zero mass in atomic units, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(B)** Silicon is an ionic crystal whose Madelung constant is negative
- [ ] **(C)** Pair potentials cannot be evaluated when $Z = 4$, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(D)** Without directional 3-body angular penalties enforcing the $109.47^\circ$ tetrahedral angle of $sp^3$ bonds, a radial pair potential always lowers its energy by packing more neighbors ($Z = 12$ close-packed FCC/HCP) into the attractive well

In [39]:
ws.render_section('q30_sw_tetrahedral_cosine')

**Q30 · easy** · 💡

In the Stillinger-Weber 3-body potential $v_3 \propto \lambda \exp\!\left[\frac{\gamma}{r_{ij}-a} + \frac{\gamma}{r_{ik}-a}\right]\left(\cos\theta_{jik} + \frac{1}{3}\right)^2$, why does the angular factor contain $+1/3$ inside the square?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $\cos(109.4712^\circ)$ for an ideal tetrahedron.

</div>
</details>

- [ ] **(A)** Because the ideal tetrahedral bond angle $\theta_{\text{tet}} = 109.47^\circ$ satisfies $\cos\theta_{\text{tet}} = -1/3$, making the penalty $(\cos\theta_{jik} + 1/3)^2$ vanish ($=0$) at the exact tetrahedral angle and strictly positive elsewhere
- [ ] **(B)** Because Silicon has 3 valence electrons per atom, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(C)** Because $\cos(60^\circ) = -1/3$ in an FCC lattice, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(D)** To cancel the $1/3!$ prefactor in the many-body cluster expansion

In [40]:
ws.render_section('q31_sw_angular_penalty_60deg')

**Q31 · medium** · 💡

In the Stillinger-Weber 3-body potential, the angular penalty factor is given by $g(\theta) = \left(\cos\theta + \frac{1}{3}\right)^2$, which vanishes at the tetrahedral angle $\theta \approx 109.47^\circ$. What is the value of this angular factor $g(\theta)$ at an angle of $\theta = 60^\circ$, characteristic of close-packed triangular configurations? (Enter a decimal to three places, e.g., `0.694`.)

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate the angular function using the cosine of an equilateral triangular bond angle.

</div>
</details>

_Numeric answer required._

In [41]:
ws.render_section('q32_sw_smooth_cutoff_exponential')

**Q32 · medium** · 💡

In both the 2-body and 3-body terms of the Stillinger-Weber potential, the radial cutoff at $r = a$ is implemented via the factor $\exp\!\left(\frac{\gamma}{r - a}\right)$ for $r < a$ (and $0$ for $r \ge a$, with $\gamma > 0$). What is the mathematical advantage of this exponential cutoff factor as $r \to a^-$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

As $r$ approaches $a$ from below ($r < a$), $r - a$ approaches $0^-$ so $\gamma / (r - a) \to -\infty$.

</div>
</details>

- [ ] **(A)** It makes the potential diverge to $+\infty$ at $r = a$ to trap neighbors inside a hard shell
- [ ] **(B)** As $r \to a^-$, $\exp(\gamma/(r-a))$ and all of its derivatives ($V, V', V'', \dots$) approach zero smoothly ($C^\infty$), completely avoiding force or derivative discontinuities at the cutoff $r = a$
- [ ] **(C)** It extends the interaction range to $r = \infty$ like a Coulomb potential
- [ ] **(D)** It avoids calling the `exp` function in computer code, which fails to satisfy the necessary boundary conditions and physical conservation laws.

In [42]:
ws.render_section('q33_sw_fitting_targets')

**Q33 · medium** · 💡

According to the principles of simulation methodology,, what set of physical properties did Stillinger and Weber (1985) use to fit their Silicon potential?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the orange bullet point.

</div>
</details>

- [ ] **(A)** The Cauchy elastic ratio $C_{12}/C_{44}$ of FCC Copper
- [ ] **(B)** Gas-phase $\text{Si}_2$ dimer spectroscopy and ionization energy only
- [ ] **(C)** Lattice constant, cohesive energy, melting point, and the structure of liquid Silicon
- [ ] **(D)** Ab initio Hartree-Fock forces of $\text{SiH}_4$ in vacuum only

In [43]:
ws.render_section('q34_amber_force_field_terms')

**Q34 · easy** · 💡

In a biomolecular/hydrocarbon force field such as **AMBER**, which combination of terms represents the intra-molecular (bonded) and inter-molecular (non-bonded) interactions?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the 4 terms listed under 'AMBER potential is:'.

</div>
</details>

- [ ] **(A)** Bonded: Born-Mayer exponential repulsion only; Non-bonded: zero
- [ ] **(B)** Bonded: EAM embedding function $F(\rho)$; Non-bonded: Stillinger-Weber 3-body potential
- [ ] **(C)** Bonded: hard-sphere collisions only; Non-bonded: harmonic springs between all pairs in the box
- [ ] **(D)** Bonded: harmonic bond stretch $k_r(r_i - r_j)^2$, harmonic bond angle $k_a(\theta - \theta_0)^2$, and periodic dihedral torsion $v_n[1 - \cos(n\phi)]$; Non-bonded: 2-body Lennard-Jones 12-6 + point-charge Coulomb interaction

In [44]:
ws.render_section('q35_ethane_dihedral_4body')

**Q35 · medium** · 💡

Why does distinguishing the **staggered** versus **eclipsed** conformation of Ethane ($\text{H}_3\text{C–CH}_3$, rotating around the central $\text{C–C}$ bond with $V_{\text{torsion}} = K\cos(3\phi)$) require a **four-body potential**?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How many atoms are needed to define a dihedral torsion angle $\phi$ between two intersecting planes?

</div>
</details>

- [ ] **(A)** Defining a torsional (dihedral) angle $\phi$ around a central bond $B\text{–}C$ requires four sequentially bonded atoms $A\text{–}B\text{–}C\text{–}D$ to specify the angle between plane $(A,B,C)$ and plane $(B,C,D)$
- [ ] **(B)** Ethane contains exactly 4 atoms in total, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(C)** A 2-body bond stretch $k_r(r_{\text{CC}} - r_0)^2$ changes strongly as the methyl group rotates at fixed $\text{C–C}$ bond length
- [ ] **(D)** The $\text{H–C–C}$ 3-body bond angle changes from $109.5^\circ$ to $180^\circ$ during rotation

In [45]:
ws.render_section('q36_ethane_vs_ethene_hybridization')

**Q36 · medium** · 💡

According to the principles of simulation methodology,, why can't a single fixed-parameter carbon potential in AMBER or CHARMM simultaneously describe both Ethane ($\text{H}_3\text{C–CH}_3$, single bond) and Ethene ($\text{H}_2\text{C=CH}_2$, double bond) without changing atom types or switching to a cluster functional?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the hybridization and bond strength of $sp^3$ vs. $sp^2$ carbon.

</div>
</details>

- [ ] **(A)** Ethene is a metal with delocalized $d$-electrons, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(B)** The $\text{C=C}$ double bond ($sp^2$) is much stiffer, shorter, and has different equilibrium angles ($120^\circ$ vs. $109.5^\circ$) than the $\text{C–C}$ single bond ($sp^3$), requiring either distinct predefined potentials/atom types for $sp, sp^2, sp^3$ carbon or a bond-order cluster functional
- [ ] **(C)** Harmonic springs $k_r(r - r_0)^2$ automatically convert from single to double bonds as $r$ decreases
- [ ] **(D)** AMBER does not include bond angle potentials, which fails to satisfy the necessary boundary conditions and physical conservation laws.

In [46]:
ws.render_section('q37_tip5p_water_architecture')

**Q37 · medium** · 💡

How does the **TIP5P** water model (Mahoney & Jorgensen, 2000; of the reference notes) represent a single $\text{H}_2\text{O}$ molecule?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Count the 5 interaction sites and check which sites carry Lennard-Jones vs. Electrostatic charges ($\pm 0.24e$).

</div>
</details>

- [ ] **(A)** 5 separate Lennard-Jones spheres with zero electrostatic charges
- [ ] **(B)** A flexible 3-site molecule with Morse O-H bonds and EAM embedding on the Oxygen atom
- [ ] **(C)** A rigid 5-site molecule: a central Oxygen site that interacts with other Oxygens via Lennard-Jones 12-6, surrounded by 4 point charges ($q = \pm 0.24e$, two positive on H sites and two negative on tetrahedral lone-pair sites)
- [ ] **(D)** A 5-body Stillinger-Weber potential without Coulomb interactions

In [47]:
ws.render_section('q38_buckingham_plus_coulomb')

**Q38 · easy** · 💡

In the discussion of Potentials for Charged Systems, the **Buckingham + Coulomb** potential is written as:
$$V(r) = A\exp\!\left(-\frac{r}{B}\right) - \frac{C}{r^6} + \frac{q_1 q_2}{r}$$
How does the Buckingham short-range part differ from a Lennard-Jones 12-6 potential?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the repulsive term of Buckingham with the $(\sigma/r)^{12}$ term of LJ.

</div>
</details>

- [ ] **(A)** It is a 4-body dihedral potential, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(B)** It replaces the $-r^{-6}$ dispersion term with a $+r^{+6}$ harmonic confinement
- [ ] **(C)** It omits Pauli repulsion entirely, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(D)** It replaces the artificial $r^{-12}$ power-law repulsion with a physically realistic Born-Mayer exponential core repulsion $A\exp(-r/B)$ while retaining the $-C/r^6$ dispersion attraction

In [48]:
ws.render_section('q39_core_shell_polarizability_model')

**Q39 · medium** · 💡

In ionic crystals where local electric fields induce dipole moments on polarizable ions, how does the **Shell Model** represent electronic polarizability within a classical simulation?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the diagram on the bottom right of: two particles connected by a spring.

</div>
</details>

- [ ] **(A)** Each polarizable ion is split into an ion core and an electron shell modeled as two point charges connected by a harmonic spring, where the spring constant $k_s$ determines the ion's polarizability ($\alpha = q_{\text{shell}}^2 / k_s$)
- [ ] **(B)** The Kohn-Sham Schrödinger equation is solved on a 3D FFT grid at every timestep
- [ ] **(C)** The ionic charge $q_i$ is multiplied by the temperature $T$, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(D)** The cutoff radius $r_c$ is varied dynamically with the local coordination number

In [49]:
ws.render_section('q40_core_shell_spring_calc')

**Q40 · hard** · 💡

In a Core-Shell model for an $\text{O}^{2-}$ anion, the valence shell carries charge $q_s = -2.0\,e$ and is connected to the core by a harmonic spring with constant $k_s = 20.0\text{ eV/Å}^2$. If a local electric field of magnitude $|\mathbf{E}| = 0.50\text{ V/Å}$ acts on the ion, by what distance $d$ (in $\text{Å}$) does the shell displace relative to the core in static equilibrium?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Balance the electrostatic force exerted on the shell charge by the local electric field against the restoring harmonic spring force.

</div>
</details>

_Numeric answer required._

In [50]:
ws.render_section('q41_inhomogeneous_environment_problems')

**Q41 · medium** · 💡

According to the principles of simulation methodology, ("Problems with potentials"), select **all** true guidelines and caveats when developing or applying empirical potentials:

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review on fitting data, testing, inhomogeneous environments, and universality.

</div>
</details>

- [ ] **(A)** Fit the potential to data relevant to the system you plan to simulate (similar densities and local atomic environments), and use independent experiments to test/validate it
- [ ] **(B)** No simple empirical potential works very well across strongly inhomogeneous environments (e.g., where bonding character changes from metallic bulk to covalent/ionic oxide at an interface)
- [ ] **(C)** Universality often helps in statistical mechanics: for generic qualitative or universal scaling phenomena, fine details of the potential may not matter as much
- [ ] **(D)** Quantum chemical (SCF or DFT) cluster calculations are guaranteed to be $100\%$ exact and never have systematic errors

In [51]:
ws.render_section('q42_potential_material_matrix')

**Q42 · easy** · 💡

Based on the summary matrix of the reference notes ("Summary of semi-empirical potentials"), which potential family is matched with its primary material class?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Match Noble Gases, Metals, Semiconductors, Polymers/Biomaterials, and Ionic Materials.

</div>
</details>

- [ ] **(A)** Lennard-Jones $\to$ Semiconductors; Stillinger-Weber $\to$ Noble Gases; EAM $\to$ Polymers; AMBER $\to$ Metals
- [ ] **(B)** Lennard-Jones $\to$ Noble Gases; EAM $\to$ Metals; Stillinger-Weber $\to$ Semiconductors; AMBER/CHARMM/MM3 $\to$ Polymers, Bio- & Organic materials; Buckingham+Coulomb / Shell $\to$ Ionic materials
- [ ] **(C)** Morse $\to$ Biomaterials only; Shell potentials $\to$ Noble Gases only
- [ ] **(D)** Buckingham+Coulomb $\to$ Pure FCC Metals; EAM $\to$ Dimers in vacuum

In [52]:
ws.render_section('q43_lammps_script_supercell_atoms')

**Q43 · medium** · 💡

Examine the sample LAMMPS input script of the reference notes:
```text
units lj
boundary p p p
atom_style atomic
lattice fcc 1.0
region box block 0 6 0 6 0 6 units lattice
create_box 1 box
create_atoms 1 box
```
Since a conventional FCC unit cell contains $4$ atoms and `region box block 0 6 0 6 0 6 units lattice` creates a $6 \times 6 \times 6$ supercell of FCC unit cells under 3D periodic boundary conditions (`p p p`), how many total atoms $N$ are created by `create_atoms 1 box`?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Determine the total number of unit cells defined by the bounding box block along the three coordinate axes and multiply by the atom count per conventional FCC unit cell.

</div>
</details>

_Numeric answer required._

In [53]:
ws.render_section('q44_lammps_script_commands')

**Q44 · easy** · 💡

In the remainder of the LAMMPS script of the reference notes:
```text
pair_style lj/cut 2.5
pair_coeff 1 1 1.0 1.0 2.5
mass 1 1.0
velocity all create 1.2 987654 rot yes dist gaussian
fix fix1 all nve
timestep 0.005
run 1000
```
Select **all** accurate interpretations of these commands:

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the comments next to each command.

</div>
</details>

- [ ] **(A)** `pair_style lj/cut 2.5` and `pair_coeff 1 1 1.0 1.0 2.5` specify a Lennard-Jones 12-6 potential with $\varepsilon = 1.0$, $\sigma = 1.0$, truncated at cutoff $r_c = 2.5\,\sigma$
- [ ] **(B)** `velocity all create 1.2 987654 rot yes dist gaussian` initializes atomic velocities from a Gaussian (Maxwell-Boltzmann) distribution at reduced temperature $T^* = 1.2$
- [ ] **(C)** `fix fix1 all nve` integrates the equations of motion at constant particle number $N$, volume $V$, and total energy $E$ (microcanonical ensemble)
- [ ] **(D)** `timestep 0.005` and `run 1000` advance the simulation for $1000$ steps, corresponding to a total reduced time of $t^* = 1000 \times 0.005 = 5.0$

In [54]:
ws.render_section('q45_many_body_cluster_expansion_prefactor')

**Q45 · medium** · 💡

In the discussion of Many-Body Potentials, the general cluster expansion of the potential energy for unrestricted sums over particle indices $i, j, k \in \{1, \dots, N\}$ is written as:
$$E = \frac{1}{2!}\sum_{i,j=1}^N V_2(\mathbf{R}_i, \mathbf{R}_j) + \frac{1}{3!}\sum_{i,j,k=1}^N V_3(\mathbf{R}_i, \mathbf{R}_j, \mathbf{R}_k) + \cdots$$
Why do the factorial denominators $\frac{1}{2!}$ and $\frac{1}{3!}$ appear in front of the unrestricted sums?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

How many permutations of $(i, j)$ or $(i, j, k)$ refer to the exact same pair or triplet of atoms when $i, j, k$ each run from $1$ to $N$?

</div>
</details>

- [ ] **(A)** To convert from Hartrees to electron-volts, which fails to satisfy the necessary boundary conditions and physical conservation laws.
- [ ] **(B)** Because the Taylor series of $\exp(-r)$ has $1/n!$ coefficients
- [ ] **(C)** To correct for overcounting permutations of identical pairs ($2! = 2$ orderings of $\{i,j\}$) and triplets ($3! = 6$ orderings of $\{i,j,k\}$) in the unrestricted sums
- [ ] **(D)** Because 3-body forces only act on every sixth timestep, which fails to satisfy the necessary boundary conditions and physical conservation laws.

In [55]:
ws.render_section('q46_selecting_simulation_methodology')

**Q46 · medium** · 💡

You need to simulate dislocation loop interactions and plastic deformation in a nanocrystalline Copper sample containing $N = 10^7$ atoms over $t = 10\text{ ns}$ ($5 \times 10^6$ timesteps). Based of the reference notes, which potential and method is the only physically appropriate and computationally feasible choice?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Copper is an FCC metal, and $N = 10^7$ atoms for $5 \times 10^6$ steps requires an $\mathcal{O}(N)$ metallic potential.

</div>
</details>

- [ ] **(A)** Stillinger-Weber tetrahedral diamond-cubic silicon potential ($Z=4$)
- [ ] **(B)** Ab initio DFT Molecular Dynamics (scales as $\mathcal{O}(N^3)$, limited to $\sim 10^2\text{–}10^3$ atoms for $\sim 10\text{–}100\text{ ps}$)
- [ ] **(C)** Lennard-Jones 12-6 pair potential (wrong $C_{12}=C_{44}$ Cauchy elasticity, wrong $E_v/E_c = 1$, and wrong surface relaxation for Cu)
- [ ] **(D)** Embedded Atom Method (EAM) with hybrid Cell + Verlet neighbor lists ($\mathcal{O}(N)$ scaling and accurate metallic vacancy/stacking-fault/elastic properties)

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()